# Signum Demo

**Professional financial charting, inspired by Lightweight Charts.**

Works seamlessly in Jupyter notebooks, Dash apps, and Streamlit dashboards — same chart object, multiple outputs.

Themes: `dark` · `light` · `ft` (Financial Times) · `midnight` · `rome` · `distfit`


In [ ]:
# Use the installed package (`pip install -e .` from the repo, or `pip install signum-charts`).
# If signum isn't importable, fall back to the in-repo source under ./src so the
# demo still runs from a fresh checkout.
try:
    from signum import Chart
except ImportError:
    import sys
    from pathlib import Path
    sys.modules.pop("signum", None)          # drop any half-resolved namespace pkg
    sys.path.insert(0, str(Path.cwd() / "src"))
    from signum import Chart

import pandas as pd
import yfinance as yf

# Download 1 year of AAPL data
df = yf.download("AAPL", period="1y", auto_adjust=True)
df = df.reset_index()
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]  # flatten multi-index

# Calculate moving averages
sma_20 = pd.DataFrame({"time": df["Date"], "value": df["Close"].rolling(20).mean()}).dropna()
sma_50 = pd.DataFrame({"time": df["Date"], "value": df["Close"].rolling(50).mean()}).dropna()

print(f"Loaded {len(df)} bars for AAPL")
df.tail(3)

## 1. Candlestick Chart with Volume
Basic OHLCV chart rendered inline in the notebook.

In [36]:
# FT theme candlestick + volume — y_format="kmb" adds the ⚙ gear icon (top-right)
# Click it to switch the volume axis between Auto / Thousands / Millions / Billions / Raw
chart = Chart(theme="ft", height=450, y_format="kmb")
chart.candlestick(df).volume(df)
chart

## 2. With Moving Average Overlays
Chain `.candlestick()`, `.line()`, `.volume()` to compose the chart. Each `.line()` auto-cycles colors.

In [37]:
# Midnight theme with SMA overlays + watermark
chart = (
    Chart(theme="midnight", height=500, watermark="AAPL")
    .candlestick(df)
    .line(sma_20, name="SMA 20", color="#FF6D00", width=1)
    .line(sma_50, name="SMA 50", color="#E91E63", width=1)
    .volume(df)
)
chart

## 3. Chart Types — Area, Baseline & Themes
Area charts for equity curves, baseline for P&L (green above / red below), and theme examples across the full palette.


In [38]:
# Area chart using close prices
Chart(theme="distfit", height=350).area(df, value_col="Close", name="AAPL")

In [39]:
# Cumulative returns as baseline chart
returns = pd.DataFrame({
    "time": df["Date"],
    "value": ((df["Close"] / df["Close"].iloc[0]) - 1) * 100
}).dropna()

Chart(theme="dark", height=350).baseline(returns, base_value=0, value_col="value")

## 4. Save & Integration

The **same chart object** works everywhere:

| Method | Where |
|--------|-------|
| `chart` or `chart.show()` | Jupyter Notebook (inline iframe) |
| `chart.save("out.html")` | Standalone HTML file |
| `chart.to_dash(id="x")` | Dash app component (`html.Iframe`) |
| `chart.to_streamlit()` | Streamlit dashboard |
| `chart.render()` | Raw HTML string for any embedding |


In [40]:
# Save as standalone HTML
chart = Chart(theme="dark", watermark="AAPL").candlestick(df).volume(df)
chart.save("aapl_chart.html")
print("✅ Saved to aapl_chart.html — open in any browser\n")

# Dash integration (copy-paste into a .py file):
print("""# ── Dash App ──────────────────────────────────
from dash import Dash, html
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)

app = Dash(__name__)
app.layout = html.Div([
    html.H1("AAPL Dashboard"),
    chart.to_dash(id="price-chart"),
])
app.run(debug=True)
""")

# Streamlit integration (copy-paste into a .py file):
print("""# ── Streamlit App ─────────────────────────────
import streamlit as st
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)
st.title("AAPL Dashboard")
chart.to_streamlit()
""")

✅ Saved to aapl_chart.html — open in any browser

# ── Dash App ──────────────────────────────────
from dash import Dash, html
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)

app = Dash(__name__)
app.layout = html.Div([
    html.H1("AAPL Dashboard"),
    chart.to_dash(id="price-chart"),
])
app.run(debug=True)

# ── Streamlit App ─────────────────────────────
import streamlit as st
from signum import Chart

chart = Chart(theme="dark").candlestick(df).volume(df)
st.title("AAPL Dashboard")
chart.to_streamlit()



## 5. Price Lines & Markers
Add horizontal price lines and markers for signals, targets, support/resistance levels.


In [41]:
# Annotated chart with price lines and signal markers
avg_price = float(df["Close"].mean())
max_price = float(df["Close"].max())
min_date = df.loc[df["Close"].idxmin(), "Date"].strftime("%Y-%m-%d")
max_date = df.loc[df["Close"].idxmax(), "Date"].strftime("%Y-%m-%d")

chart = (
    Chart(theme="midnight", height=450, watermark="AAPL")
    .candlestick(df)
    .volume(df)
    .price_line(avg_price, title="Average", color="#FF9800")
    .marker(min_date, text="Low", position="belowBar", shape="arrowUp", color="#26a69a")
    .marker(max_date, text="High", position="aboveBar", shape="arrowDown", color="#ef5350")
)
chart

## 6. Buy / Sell Signals with Position Shading
Buy/sell arrows via `.signals()` plus `.shade()` to highlight periods when the strategy is "in the deal". The shaded background makes it easy to see at a glance when you're invested.


In [ ]:
from signum import Chart
import numpy as np

# ── Artificial buy/sell signals at fixed dates ────────────────────
# (In practice these come from your model — here we just pick dates)
buy_dates  = df["Date"].iloc[np.linspace(30, len(df)-60, 4, dtype=int)].values
sell_dates = df["Date"].iloc[np.linspace(60, len(df)-30, 4, dtype=int)].values

df["signal"] = 0
df.loc[df["Date"].isin(buy_dates), "signal"] = 1
df.loc[df["Date"].isin(sell_dates), "signal"] = -1

# Build position column: 1 between buy and sell, 0 otherwise
df["position"] = 0
in_pos = False
for i in range(len(df)):
    if df["signal"].iloc[i] == 1:
        in_pos = True
    elif df["signal"].iloc[i] == -1:
        in_pos = False
    df.iloc[i, df.columns.get_loc("position")] = 1 if in_pos else 0

n_buys = (df["signal"] == 1).sum()
n_sells = (df["signal"] == -1).sum()
print(f"{n_buys} BUY / {n_sells} SELL signals  •  In market {df['position'].mean()*100:.0f}% of the time")

# ── Chart with arrows + shaded background during position ─────────
chart = (
    Chart(theme="dark", height=500, watermark="AAPL")
    .candlestick(df)
    .volume(df)
    .signals(df, signal_col="signal")
    .shade(df, position_col="position")
)
chart

## 7. Multi-Pane Synchronized Dashboard
Vertically stacked panes with synchronized crosshair, zoom, and scroll. One `theme=` themes **all panes** at once.

A 4-pane **Time in Market** breakdown: price + position shading, continuous signal, binary position histogram, and strategy vs buy & hold equity.

In [ ]:
from signum import Chart, Dashboard

# ── Compute continuous signal + position from SMA crossover ───────
sma20 = df["Close"].rolling(20).mean()
sma50 = df["Close"].rolling(50).mean()

# Continuous signal: SMA spread % (tracks signal buildup/decay)
signal_vals = ((sma20 - sma50) / sma50 * 100)

# Position: +1 when SMA20 > SMA50, else 0
position = (sma20 > sma50).astype(float)

# Equity curves
daily_ret = df["Close"].pct_change()
strat_ret = daily_ret * position.shift(1)
strategy_eq = (1 + strat_ret.fillna(0)).cumprod() * 100
bh_eq = (1 + daily_ret.fillna(0)).cumprod() * 100

# ── Align to common date range ────────────────────────────────────
mask = signal_vals.notna() & position.notna()
dates = df.loc[mask, "Date"]
df_aligned = df[mask].copy()
df_aligned["position"] = position[mask].values

# Re-index both equity curves to start at 100 from the aligned start
strat_aligned = strategy_eq[mask].values
bh_aligned = bh_eq[mask].values
strat_aligned = strat_aligned / strat_aligned[0] * 100
bh_aligned = bh_aligned / bh_aligned[0] * 100

sig_df  = pd.DataFrame({"time": dates, "value": signal_vals[mask].values})
pos_df  = pd.DataFrame({"time": dates, "value": position[mask].values})
eq_df   = pd.DataFrame({"time": dates, "value": strat_aligned})
bh_df   = pd.DataFrame({"time": dates, "value": bh_aligned})

# Generate crossover signals for markers
cross_above = (sma20[mask].values > sma50[mask].values) & \
              (pd.Series(sma20[mask].values).shift(1).values <= pd.Series(sma50[mask].values).shift(1).values)
cross_below = (sma20[mask].values < sma50[mask].values) & \
              (pd.Series(sma20[mask].values).shift(1).values >= pd.Series(sma50[mask].values).shift(1).values)
df_aligned["signal"] = 0
df_aligned.loc[df_aligned.index[cross_above], "signal"] = 1
df_aligned.loc[df_aligned.index[cross_below], "signal"] = -1

# ── Stats ─────────────────────────────────────────────────────────
time_in_market = position[mask].mean() * 100
print(f"Time in Market: {time_in_market:.1f}%  ({int(position[mask].sum())}/{len(df_aligned)} days)")
print(f"Strategy Return: {strategy_eq[mask].iloc[-1] - 100:+.1f}%  |  Buy & Hold: {bh_eq[mask].iloc[-1] - 100:+.1f}%")

# ── Build 4-pane dashboard ────────────────────────────────────────
price_pane = (
    Chart(height=280)
    .candlestick(df_aligned)
    .volume(df_aligned)
    .signals(df_aligned, signal_col="signal")
    .shade(df_aligned, position_col="position")
)

signal_pane = Chart(height=120).baseline(sig_df, base_value=0)

position_pane = (
    Chart(height=90)
    .histogram(pos_df, name="Position", color="rgba(38, 166, 154, 0.7)")
)

equity_pane = (
    Chart(height=170)
    .area(eq_df, name="Strategy", color="#26a69a", line=True, tag=True)
    .line(bh_df, name="Buy & Hold", color="#5c6bc0", width=1, line=True, tag=True)
)

dash = Dashboard(
    panes=[price_pane, signal_pane, position_pane, equity_pane],
    titles=[
        "AAPL — Price + Position Shading",
        "Signal (SMA Spread %) — continuous evolution",
        f"Position (Time in Market: {time_in_market:.0f}%)",
        "Equity — Strategy vs Buy & Hold (indexed 100)",
    ],
)
dash

## 8. Statistical Charts — Distributions & Scatter
Native Canvas-based statistical visualisations. `StatChart` follows the same API: theme-aware, fluent chaining, same output pipeline.

| Method | Panel Type |
|--------|-----------|
| `.distribution(data)` | Histogram with mean/median lines + optional KDE fit |
| `.scatter(x, y)` | XY scatter plot |


In [ ]:
from signum import StatChart
import numpy as np

np.random.seed(42)

# Simulate volatility features (similar to IVol Z-Score, IVol Level, Vol Spread)
ivol_zscore = np.random.standard_normal(1000) * 0.8 - 0.3      # skewed left
ivol_level  = np.random.lognormal(2.8, 0.35, 1000)              # right-skewed, ~15-25 range
vol_spread  = np.random.normal(4, 3.5, 1000)                     # centered ~4

# ── Side-by-side distribution grid ────────────────────────────────
stat = (
    StatChart(theme="distfit", height=320, cols=3,
              title="Volatility Feature Distributions (2015–present)")
    .distribution(ivol_zscore, name="IVol Z-Score Distribution")
    .distribution(ivol_level,  name="IVol Level Distribution",   color="#E91E63")
    .distribution(vol_spread,  name="Vol Spread Distribution",   color="#9C27B0")
)
stat

In [45]:
# ── Scatter plot: Returns vs Volume ───────────────────────────────
daily_ret = df["Close"].pct_change().dropna().values * 100
daily_vol = df["Volume"].iloc[1:].values / 1e6

(StatChart(theme="ft", height=380, title="Returns vs Volume")
    .scatter(daily_vol, daily_ret, name="Daily Return % vs Volume (M)", size=3)
)

## 8b. Yield Curve — fitted line, confidence band & date slider

`.curve()` draws a term-structure panel: observed dots + a smoothed posterior line + a shaded confidence band, plus an optional second curve (e.g. an NSS prior) and a pinned **base-date ghost**. Pass per-date `frames` to get a **date slider** that morphs the whole curve through history. `.spread()` adds a linked panel below showing `active − base` per maturity — driven by the same slider.

| Method | Panel Type |
|--------|-----------|
| `.curve(x, mean=, lower=, upper=, points=, prior=)` | Points + fitted line + 95% band (+ 2nd curve) |
| `.curve(x, frames={date: {...}}, base=)` | …with a date slider + pinned base-date ghost |
| `.spread(x, frames=, base=)` | Difference panel: `active − base`, vs a zero line |

A synthetic sovereign curve below — all data synthetic (NSS + smooth wiggles + noise). **Drag the slider** to watch the curve evolve and the spread fill in.

In [ ]:
from signum import StatChart
import numpy as np, pandas as pd

# ── Synthetic sovereign curve: NSS prior + GP-style posterior ──────────────
def nss(t, b0, b1, b2, b3, tau1, tau2):
    t = np.where(np.asarray(t, float) <= 0, 1e-6, np.asarray(t, float))
    f1 = (1 - np.exp(-t / tau1)) / (t / tau1)
    f2 = f1 - np.exp(-t / tau1)
    f3 = (1 - np.exp(-t / tau2)) / (t / tau2) - np.exp(-t / tau2)
    return b0 + b1 * f1 + b2 * f2 + b3 * f3

rng       = np.random.default_rng(7)
grid      = np.linspace(0.25, 15.0, 220)            # ttm grid (years)
bond_ttm  = np.sort(rng.uniform(0.3, 14.5, 30))     # observed bond maturities
dates     = pd.date_range("2023-06-01", "2026-06-01", freq="MS")

frames = {}
for k, dt in enumerate(dates):
    w = k / (len(dates) - 1)                          # 0 → 1 over time
    prior = nss(grid, 14.3 + 0.5*w, -1.4 + 1.0*w, 1.2 - 2.2*w, -0.8 + 0.4*w, 1.6, 5.5)
    mean  = prior + 0.18*np.sin(grid/1.3 + 1.5*w*np.pi) + 0.10*np.sin(grid/3.7 + 0.8*k)
    edge  = np.clip((grid - bond_ttm.max()) / 2.0, 0, None)
    sd    = 0.05 + 0.015*np.sqrt(grid) + 0.55*edge**1.5      # band flares past last bond
    py    = np.interp(bond_ttm, grid, mean) + rng.normal(0, 0.06, bond_ttm.shape)
    frames[dt.strftime("%Y-%m-%d")] = {
        "mean": mean, "lower": mean - 1.96*sd, "upper": mean + 1.96*sd,
        "prior": prior, "points": (bond_ttm, py),
    }

base = list(frames)[0]   # pin earliest date as ghost; slider starts on latest

# ── curve panel + linked spread panel, one shared date slider ──────────────
# Colours come from the theme palette — try "midnight", "dark", "distfit" too.
(
    StatChart(theme="ft", height=720, cols=1,
              title="Sovereign curve — NSS prior + GP smoothing (drag the date slider)")
    .curve(
        grid, frames=frames, base=base,
        mean_name="GP posterior mean", prior_name="NSS prior",
        point_name="sovereign bonds", base_name="base date", band_label="GP 95% band",
        name="Effective yield (%) vs time to maturity",
        x_label="Time to maturity (years)", y_label="Effective yield (%)",
        slider_label="curve date",
    )
    .spread(
        grid, frames=frames, base=base,
        name="Spread vs base date  (active − base, %)",
        x_label="Time to maturity (years)", y_label="Δ yield (%)",
    )
)

## 9. Interactive Backtest Dashboard — Live Slider with Equity & Stats

Drag **θ** and everything updates instantly (pure JS, no Python roundtrip):

- **Price pane** — candlestick + buy/sell arrows reposition live
- **Signal pane** — continuous signal with moving θ line, coloured above/below threshold
- **Equity pane** — strategy vs buy & hold curve rebuilds on every tick, with a **stats overlay** (Return, B&H, CAGR, Sharpe, Max DD, Win Rate, Time in Market)

Supports multiple signal modes: `>=`, `>`, `<=`, `<`, `"rising"`, `"crossover"`, `"within"`, `"outside"` — see docstring for details.

In [ ]:
from signum import Chart, Dashboard
import numpy as np

# ── Synthetic momentum signal (blend of fast/slow SMA spreads) ────
sma5  = df["Close"].rolling(5).mean()
sma20 = df["Close"].rolling(20).mean()
sma50 = df["Close"].rolling(50).mean()

signal_vals = ((sma5 - sma20) / sma20 * 0.6 + (sma20 - sma50) / sma50 * 0.4)
daily_ret   = df["Close"].pct_change()

mask    = signal_vals.notna()
df_t    = df[mask].copy()
sig_vals = signal_vals[mask]
daily_r  = daily_ret[mask]

pred_df = pd.DataFrame({"time": df_t["Date"], "value": sig_vals.values})
sig_min = round(float(pred_df["value"].min()), 3)
sig_max = round(float(pred_df["value"].max()), 3)
print(f"Signal range: {sig_min:.4f} → {sig_max:.4f}  ({len(df_t)} bars)")
print("signal_mode='>='  — long when signal ≥ θ  (drag slider to explore)")

# ── 3-pane backtest dashboard: signal_mode=">=" ───────────────────
price_pane  = Chart(height=280).candlestick(df_t).volume(df_t)
signal_pane = Chart(height=120).baseline(pred_df, base_value=0)
equity_pane = (
    Chart(height=200)
    .area(pd.DataFrame({"time": df_t["Date"], "value": np.ones(len(df_t))}), name="Strategy",   color="#26a69a")
    .line(pd.DataFrame({"time": df_t["Date"], "value": np.ones(len(df_t))}), name="Buy & Hold", color="#5c6bc0", width=1)
)

dash = (
    Dashboard(
        panes=[price_pane, signal_pane, equity_pane],
        titles=[
            "AAPL — Price + Signals",
            "Signal (momentum blend — green ≥ θ, grey < θ)",
            "Equity — Strategy vs Buy & Hold (indexed 100)",
        ],
        theme="ft",
    )
    .threshold_control(
        pred_df,
        signal_mode=">=",             # long when signal ≥ θ
        threshold=0.002,
        min_val=sig_min,
        max_val=sig_max,
        step=0.001,
        price_pane=0,
        daily_returns=daily_r.reset_index(drop=True),
        bh_returns=daily_r.reset_index(drop=True),
        equity_pane=2,
    )
)
dash

## 10. Live Smoothing Control

Adjust smoothing parameters in real-time without rerunning Python — JS computes SMA/EMA on drag.

In [ ]:
from signum import Chart

close = df["Close"]
dates = df["Date"]
raw_df = pd.DataFrame({"time": dates, "value": close})

# ── 10a: SMA slider — continuous, pure JS ────────────────────────────────────
win_init    = 21
sma_init_df = pd.DataFrame({"time": dates, "value": close.rolling(win_init).mean()}).dropna()

(
    Chart(theme="dark", height=280)
    .line(raw_df,      name="Close (raw)",   color="#555555", width=1)
    .line(sma_init_df, name=f"SMA {win_init}", color="#f4a261", width=2)
    .smoothing_control(
        raw_series   = close.set_axis(dates),
        series_index = -1,
        mode         = "rolling",
        window_init  = win_init,
        window_min   = 2,
        window_max   = 200,
        color        = "#f4a261",
        label        = "sma",
    )
)

## 11. New Features — `.bar()`, `.hline()`, `.allocation()`

OHLC bars as an alternative to candles, horizontal reference lines, and long/short portfolio allocation stacked inside a dashboard.

In [ ]:
# ── 11a. OHLC bar charts (alternative to candlesticks) ────────────
from signum import Chart

chart_bar = Chart(theme="dark", height=400).bar(df).volume(df)
chart_bar

In [50]:
# ── 11b. Horizontal reference lines with .hline() ────────────────
chart_hline = (
    Chart(theme="dark", height=400)
    .line(df, value_col="Close")
    .hline(170, label="Support", color="#26a69a", style=2)
    .hline(210, label="Resistance", color="#ef5350", style=2)
    .hline(avg_price, label=f"Avg: ${avg_price:.2f}", color="#2196f3", style=1)
)
chart_hline

In [51]:
# ── 11c. Portfolio allocation in a Dashboard (synchronized time periods) ──
# Smooth portfolio evolution with gradual rebalancing (long/short, leverage).
# Auto-stacks longs upward and shorts downward from 0; hover the allocation pane for weights.
from signum import Chart, Dashboard
import pandas as pd, numpy as np

np.random.seed(42)
alloc_dates = df["Date"].values
assets = ["AAPL", "GOOGL", "MSFT", "TSLA"]

# Start with equal weights, gradually adjust every 20-30 days
weights = {a: 25.0 for a in assets}
alloc_data = []

for i in range(len(alloc_dates)):
    if i % np.random.randint(20, 31) == 0 and i > 0:
        for asset in np.random.choice(assets, np.random.choice([1, 2]), replace=False):
            weights[asset] = np.clip(weights[asset] + np.random.uniform(-30, 30), -40, 100)
    alloc_data.append(weights.copy())

alloc_df_aligned = pd.DataFrame(alloc_data, index=alloc_dates).reset_index()
alloc_df_aligned.columns = ["Date"] + assets

price_pane = Chart(height=250).candlestick(df).volume(df)
alloc_pane = Chart(height=200).allocation(alloc_df_aligned)

dash = Dashboard(
    panes=[price_pane, alloc_pane],
    titles=["AAPL Price", "Portfolio Allocation (hover for weights)"],
    theme="ft",
)
dash